# Notebook 04 — Retrain the reconstructor on verbalizer outputs

Notebook 03 produced **FVE = -0.27** for the AV → AR pipeline — *worse* than the placeholder baseline (-0.05). The likely cause is a **distribution mismatch**: AR was trained on clean first-16-token placeholders, but AV generates statistically different text. AR is then asked to interpret inputs it has never seen.

**Fix:** retrain a fresh AR head on `(AV-generated text, h)` pairs. This is a one-sided substitute for the paper's joint AV/AR RL training — we don't update AV, but we close the gap on the AR side.

If this works → FVE should go positive. If it doesn't → we have a clean finding to report: *staged supervised training cannot fully recover what joint training delivers.*

## 0. Environment

In [ ]:
%pip install -q transformers==4.46.0 accelerate matplotlib "sympy>=1.13.3"

import os
MARKER = '/tmp/.kth_nla_kernel_restarted'
if not os.path.exists(MARKER):
    open(MARKER, 'w').close()
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)
else:
    print('Packages installed; kernel already restarted.')

In [ ]:
import json, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/kth-nla-2026')
    print(f'persistent storage: {DATA_ROOT}')
except ImportError:
    DATA_ROOT = Path('.')

## 1. Config + paths

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-160m'
LAYER = 6

ACT_PT  = DATA_ROOT / 'data/activations/pythia160m_layer6_wikitext2_5000.pt'
AV_CKPT = DATA_ROOT / 'checkpoints/verbalizer.pt'
AR_BASELINE = DATA_ROOT / 'checkpoints/reconstructor_head_baseline.pt'
AR_OUT  = DATA_ROOT / 'checkpoints/reconstructor_head_av_retrain.pt'
AV_DESCS = DATA_ROOT / 'data/av_descriptions.json'  # cache so we don't regenerate
FIG_DIR = DATA_ROOT / 'figures'

DESC_TOKENS = 16
TRAIN_N, VAL_N, TEST_N = 4000, 500, 500
BATCH_GEN = 32        # batch for AV generation
BATCH_TRAIN = 32      # batch for AR training
AR_EPOCHS = 5
AR_LR = 1e-3
TEMP = 1.0
TOP_K = 50

## 2. Load activations and recreate the same split

In [ ]:
acts = torch.load(ACT_PT, map_location='cpu')
N = acts.shape[0]
perm = torch.randperm(N, generator=torch.Generator().manual_seed(SEED)).tolist()
split = {
    'train': perm[:TRAIN_N],
    'val':   perm[TRAIN_N:TRAIN_N + VAL_N],
    'test':  perm[TRAIN_N + VAL_N:TRAIN_N + VAL_N + TEST_N],
}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
print(f'loaded {N} activations')

## 3. Load the trained verbalizer + a frozen base for the reconstructor

Two Pythia instances again (~1.3 GB on GPU).

In [ ]:
av_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
ar_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
for p in ar_base.parameters(): p.requires_grad_(False)
HIDDEN = av_base.config.hidden_size

class Verbalizer(nn.Module):
    def __init__(self, base_model, hidden):
        super().__init__()
        self.base = base_model
        self.embed = base_model.get_input_embeddings()
        self.act_proj = nn.Linear(hidden, hidden)
    @torch.no_grad()
    def generate(self, h, max_new_tokens, temperature=1.0, top_k=50):
        self.eval()
        act_embed = self.act_proj(h).unsqueeze(1)
        inputs_embeds = act_embed
        generated = []
        for _ in range(max_new_tokens):
            out = self.base(inputs_embeds=inputs_embeds)
            logits = out.logits[:, -1, :] / max(temperature, 1e-6)
            if top_k:
                v, _ = logits.topk(top_k)
                logits[logits < v[:, -1:].expand_as(logits)] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(-1)
            generated.append(next_id)
            inputs_embeds = torch.cat([inputs_embeds, self.embed(next_id).unsqueeze(1)], dim=1)
        return torch.stack(generated, dim=1)

av = Verbalizer(av_base, HIDDEN).to(device)
av_ckpt = torch.load(AV_CKPT, map_location=device)
av.load_state_dict(av_ckpt['state_dict'])
av.eval()
print('AV loaded.')

## 4. Pre-generate AV descriptions for all 5000 activations

Cached to JSON so re-runs are fast. Takes ~3–5 minutes on T4 the first time.

In [ ]:
if AV_DESCS.exists():
    with open(AV_DESCS, 'r', encoding='utf-8') as f:
        all_descs = json.load(f)
    print(f'loaded {len(all_descs)} cached AV descriptions from {AV_DESCS}')
else:
    print('Generating AV descriptions for all 5000 activations...')
    all_descs = [None] * N
    for start in range(0, N, BATCH_GEN):
        end = min(start + BATCH_GEN, N)
        h = acts[start:end].to(device)
        ids = av.generate(h, max_new_tokens=DESC_TOKENS, temperature=TEMP, top_k=TOP_K)
        for k, gid in enumerate(ids):
            all_descs[start + k] = tok.decode(gid, skip_special_tokens=True)
        if (end // BATCH_GEN) % 20 == 0:
            print(f'  generated {end:,} / {N:,}')
    AV_DESCS.parent.mkdir(parents=True, exist_ok=True)
    with open(AV_DESCS, 'w', encoding='utf-8') as f:
        json.dump(all_descs, f, ensure_ascii=False)
    print(f'saved to {AV_DESCS}')

print('example AV description:', repr(all_descs[split["test"][0]]))

## 5. Reconstructor (fresh head) trained on AV descriptions

In [ ]:
class Reconstructor(nn.Module):
    def __init__(self, base_model, layer, hidden):
        super().__init__()
        self.base = base_model
        self.captured = {}
        base_model.gpt_neox.layers[layer].register_forward_hook(self._hook)
        self.head = nn.Linear(hidden, hidden)
        with torch.no_grad():
            self.head.weight.copy_(torch.eye(hidden))
            self.head.bias.zero_()
    def _hook(self, m, i, o):
        self.captured['h'] = o[0] if isinstance(o, tuple) else o
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            _ = self.base(input_ids=input_ids, attention_mask=attention_mask)
        h_seq = self.captured['h']
        last_idx = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(h_seq.size(0), device=h_seq.device)
        return self.head(h_seq[batch_idx, last_idx])

ar = Reconstructor(ar_base, LAYER, HIDDEN).to(device)
print(f'fresh AR; head trainable params: {sum(p.numel() for p in ar.head.parameters()):,}')

In [ ]:
class PairDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        return {'h': acts[j], 'z': all_descs[j]}

def collate(batch):
    return {'h': torch.stack([b['h'] for b in batch]), 'z': [b['z'] for b in batch]}

loaders = {
    name: DataLoader(PairDataset(ids), batch_size=BATCH_TRAIN,
                     shuffle=(name == 'train'), collate_fn=collate)
    for name, ids in split.items()
}

def fve(p, t):
    mse = ((p - t) ** 2).sum(dim=1).mean()
    var = ((t - t.mean(0)) ** 2).sum(dim=1).mean()
    return (1.0 - mse / var).item()

@torch.no_grad()
def evaluate(loader):
    ar.eval()
    preds, trues = [], []
    for batch in loader:
        enc = tok(batch['z'], padding=True, truncation=True, max_length=DESC_TOKENS,
                  return_tensors='pt').to(device)
        h_hat = ar(enc.input_ids, enc.attention_mask).cpu()
        preds.append(h_hat); trues.append(batch['h'])
    p = torch.cat(preds); t = torch.cat(trues)
    return {
        'fve': fve(p, t),
        'cos': F.cosine_similarity(p, t, dim=1).mean().item(),
    }

## 6. Train AR on AV-generated descriptions

In [ ]:
optim = torch.optim.AdamW(ar.head.parameters(), lr=AR_LR)
history = []

for epoch in range(1, AR_EPOCHS + 1):
    ar.train()
    running, steps = 0.0, 0
    for batch in loaders['train']:
        enc = tok(batch['z'], padding=True, truncation=True, max_length=DESC_TOKENS,
                  return_tensors='pt').to(device)
        h_hat = ar(enc.input_ids, enc.attention_mask)
        h_target = batch['h'].to(device)
        loss = F.mse_loss(h_hat, h_target, reduction='mean') * HIDDEN
        optim.zero_grad(); loss.backward(); optim.step()
        running += loss.item(); steps += 1
    val = evaluate(loaders['val'])
    history.append({'epoch': epoch, 'train_mse': running/steps, **{f'val_{k}': v for k, v in val.items()}})
    print(f'epoch {epoch}: train_mse={running/steps:8.3f}  val_FVE={val["fve"]:+.4f}  val_cos={val["cos"]:+.4f}')

## 7. Test-set headline

In [ ]:
test_after = evaluate(loaders['test'])
baseline_ckpt = torch.load(AR_BASELINE, map_location=device)
baseline_fve = baseline_ckpt['test_metrics']['fve']
baseline_cos = baseline_ckpt['test_metrics']['cos']
pipeline_fve_before = av_ckpt['test_pipeline']['fve']
pipeline_cos_before = av_ckpt['test_pipeline']['cos']

print('====  COMPARISON  ====')
print(f"  Baseline (placeholder → AR_baseline)        FVE={baseline_fve:+.4f}  cos={baseline_cos:+.4f}")
print(f"  Pipeline before AR-retrain (AV → AR_base)   FVE={pipeline_fve_before:+.4f}  cos={pipeline_cos_before:+.4f}")
print(f"  Pipeline after  AR-retrain (AV → AR_retrain) FVE={test_after['fve']:+.4f}  cos={test_after['cos']:+.4f}")

## 8. Save the retrained AR head + a comparison figure

In [ ]:
torch.save({
    'state_dict': ar.head.state_dict(),
    'config': {'model_name': MODEL_NAME, 'layer': LAYER, 'hidden': HIDDEN,
               'desc_tokens': DESC_TOKENS, 'epochs': AR_EPOCHS, 'lr': AR_LR,
               'seed': SEED, 'temperature': TEMP, 'top_k': TOP_K},
    'test_metrics': test_after,
    'history': history,
}, AR_OUT)
print('saved', AR_OUT)

import matplotlib.pyplot as plt
labels = ['baseline\n(placeholder→AR_base)', 'pipeline before\n(AV→AR_base)', 'pipeline after\n(AV→AR_retrain)']
fves = [baseline_fve, pipeline_fve_before, test_after['fve']]
coss = [baseline_cos, pipeline_cos_before, test_after['cos']]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(labels, fves, color=['tab:gray','tab:red','tab:green']); axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_ylabel('FVE'); axes[0].set_title('Fraction of Variance Explained')
axes[1].bar(labels, coss, color=['tab:gray','tab:red','tab:green'])
axes[1].set_ylabel('cosine'); axes[1].set_title('Mean cosine similarity')
for ax in axes: ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.savefig(FIG_DIR / '04_pipeline_comparison.png', dpi=120)
plt.show()